# Smart Inventory — Optuna Hyperparameter Optimization for CatBoost

This notebook replaces the small manual CatBoost hyperparameter search with **Optuna**.

## Goal

Find a strong CatBoost configuration automatically by minimizing **validation MAE**.

### Optimization workflow

```text
Training data
     ↓
CatBoost
     ↓
Validation MAE
     ↓
Optuna chooses next parameters
     ↓
Repeat
     ↓
Best parameters
     ↓
Final evaluation on untouched test data
```

### Important
- The **test set is never used by Optuna**.
- Categorical variables are handled natively by CatBoost.
- The target is `Units Sold`.
- `Demand Forecast` and `Units Ordered` are excluded.
- This notebook uses the same chronological split as the previous notebooks.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import joblib

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print("Basic libraries imported!")

## 1. Install / import Optuna

If Optuna is not installed, run the installation cell once.

In [ ]:
# Run this cell if Optuna is not installed.
# !pip install optuna

import optuna

print("Optuna version:", optuna.__version__)

## 2. Load and prepare the dataset

In [ ]:
df = pd.read_csv("../DATA/retail_store_inventory.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

group_cols = ["Store ID", "Product ID"]

# Calendar features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek
df["Week_of_Year"] = df["Date"].dt.isocalendar().week.astype(int)

# Lag features
df["Lag_1"] = df.groupby(group_cols)["Units Sold"].shift(1)
df["Lag_7"] = df.groupby(group_cols)["Units Sold"].shift(7)
df["Lag_14"] = df.groupby(group_cols)["Units Sold"].shift(14)
df["Lag_28"] = df.groupby(group_cols)["Units Sold"].shift(28)

# Leakage-safe rolling features
df["Rolling_Mean_7"] = (
    df.groupby(group_cols)["Units Sold"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df["Rolling_Mean_14"] = (
    df.groupby(group_cols)["Units Sold"]
      .transform(lambda x: x.shift(1).rolling(14).mean())
)

df["Rolling_Mean_28"] = (
    df.groupby(group_cols)["Units Sold"]
      .transform(lambda x: x.shift(1).rolling(28).mean())
)

df["Rolling_Std_7"] = (
    df.groupby(group_cols)["Units Sold"]
      .transform(lambda x: x.shift(1).rolling(7).std())
)

df["Price_Difference"] = (
    df["Price"] - df["Competitor Pricing"]
)

history_features = [
    "Lag_1", "Lag_7", "Lag_14", "Lag_28",
    "Rolling_Mean_7", "Rolling_Mean_14",
    "Rolling_Mean_28", "Rolling_Std_7"
]

df_model = df.dropna(
    subset=history_features
).copy()

print("Model shape:", df_model.shape)
print("Date range:", df_model["Date"].min(), "→", df_model["Date"].max())

## 3. Define features and target

In [ ]:
categorical_features = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality"
]

numerical_features = [
    "Inventory Level",
    "Price",
    "Discount",
    "Holiday/Promotion",
    "Competitor Pricing",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Week_of_Year",
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_28",
    "Rolling_Std_7",
    "Price_Difference"
]

feature_columns = categorical_features + numerical_features
target = "Units Sold"

X = df_model[feature_columns].copy()
y = df_model[target].copy()

for col in categorical_features:
    X[col] = X[col].astype(str)

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)
print("Total features:", len(feature_columns))

## 4. Chronological train / validation / test split

In [ ]:
unique_dates = np.sort(df_model["Date"].unique())

n_dates = len(unique_dates)

train_end_idx = int(n_dates * 0.70)
validation_end_idx = int(n_dates * 0.85)

train_end_date = unique_dates[train_end_idx - 1]
validation_end_date = unique_dates[validation_end_idx - 1]

train_mask = df_model["Date"] <= train_end_date
val_mask = (
    (df_model["Date"] > train_end_date) &
    (df_model["Date"] <= validation_end_date)
)
test_mask = df_model["Date"] > validation_end_date

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

cat_feature_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_features
]

print("TRAIN:", X_train.shape, y_train.shape)
print("VALIDATION:", X_val.shape, y_val.shape)
print("TEST:", X_test.shape, y_test.shape)

print("\nTRAIN:", df_model.loc[train_mask, "Date"].min(), "→", df_model.loc[train_mask, "Date"].max())
print("VALIDATION:", df_model.loc[val_mask, "Date"].min(), "→", df_model.loc[val_mask, "Date"].max())
print("TEST:", df_model.loc[test_mask, "Date"].min(), "→", df_model.loc[test_mask, "Date"].max())

## 5. Define the Optuna objective

Optuna will search the hyperparameter space and minimize validation MAE.

In [ ]:
def objective(trial):
    params = {
        "iterations": trial.suggest_int(
            "iterations", 500, 2000, step=100
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.10, log=True
        ),
        "depth": trial.suggest_int(
            "depth", 4, 10
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1.0, 20.0, log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 2.0
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0.0, 5.0
        )
    }

    model = CatBoostRegressor(
        **params,
        loss_function="MAE",
        eval_metric="MAE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )

    val_predictions = model.predict(X_val)

    validation_mae = mean_absolute_error(
        y_val,
        val_predictions
    )

    return validation_mae

print("Optuna objective created!")

## 6. Run the optimization

Start with 30 trials. You can increase this later to 50–100 if the results continue improving.

**Expected runtime:** depends on your CPU and CatBoost training speed.

In [ ]:
study = optuna.create_study(
    direction="minimize",
    study_name="catboost_demand_forecasting"
)

N_TRIALS = 30

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print("\nOptimization complete!")
print("Trials:", len(study.trials))
print("Best validation MAE:", study.best_value)
print("Best parameters:")
print(study.best_params)

## 7. Inspect the best trials

In [ ]:
trials_df = study.trials_dataframe(
    attrs=("number", "value", "params", "state")
)

trials_df = trials_df.sort_values(
    "value",
    ascending=True
)

trials_df.head(10)

## 8. Visualize optimization history

In [ ]:
import matplotlib.pyplot as plt

completed_trials = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
]

trial_numbers = [trial.number for trial in completed_trials]
trial_values = [trial.value for trial in completed_trials]

plt.figure(figsize=(10, 5))
plt.plot(trial_numbers, trial_values, marker="o")
plt.xlabel("Trial")
plt.ylabel("Validation MAE")
plt.title("Optuna Optimization History")
plt.tight_layout()
plt.show()

## 9. Train the best CatBoost configuration

In [ ]:
best_params = study.best_params.copy()

best_catboost = CatBoostRegressor(
    **best_params,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

best_catboost.fit(
    X_train,
    y_train,
    cat_features=cat_feature_indices,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100
)

print("Best model trained!")

## 10. Evaluate on validation and test

In [ ]:
def evaluate_predictions(name, y_true, predictions):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "R2": r2_score(y_true, predictions)
    }

best_val_pred = best_catboost.predict(X_val)
best_test_pred = best_catboost.predict(X_test)

optimized_val = evaluate_predictions(
    "Optuna CatBoost - Validation",
    y_val,
    best_val_pred
)

optimized_test = evaluate_predictions(
    "Optuna CatBoost - Test",
    y_test,
    best_test_pred
)

print("VALIDATION")
print(optimized_val)

print("\nTEST")
print(optimized_test)

## 11. Compare with the original CatBoost benchmark

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "Original One-Hot CatBoost",
        "Test MAE": 68.924640,
        "Test RMSE": 87.909215,
        "Test R2": 0.339560
    },
    {
        "Model": "Optuna Native CatBoost",
        "Test MAE": optimized_test["MAE"],
        "Test RMSE": optimized_test["RMSE"],
        "Test R2": optimized_test["R2"]
    }
])

comparison

## 12. Calculate improvement

In [ ]:
baseline_mae = 68.924640
optimized_mae = optimized_test["MAE"]

improvement = (
    (baseline_mae - optimized_mae)
    / baseline_mae
) * 100

print(f"Baseline MAE:   {baseline_mae:.4f}")
print(f"Optimized MAE:  {optimized_mae:.4f}")
print(f"MAE improvement: {improvement:.2f}%")

## 13. Feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_catboost.get_feature_importance()
}).sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(20)

In [ ]:
top_features = feature_importance.head(20).sort_values("Importance")

plt.figure(figsize=(10, 7))
plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)
plt.title("Top Optuna CatBoost Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 14. Save the optimized model and Optuna study

In [ ]:
models_dir = Path("../Models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "catboost_optuna_demand_model.cbm"
best_catboost.save_model(model_path)

joblib.dump(
    {
        "feature_columns": feature_columns,
        "categorical_features": categorical_features,
        "numerical_features": numerical_features,
        "target_column": target
    },
    models_dir / "catboost_optuna_feature_metadata.pkl"
)

joblib.dump(
    study,
    models_dir / "catboost_optuna_study.pkl"
)

trials_df.to_csv(
    models_dir / "catboost_optuna_trials.csv",
    index=False
)

print("Saved:")
print(model_path)
print(models_dir / "catboost_optuna_feature_metadata.pkl")
print(models_dir / "catboost_optuna_study.pkl")
print(models_dir / "catboost_optuna_trials.csv")

# Final interpretation

After running this notebook:

### If Optuna improves the test MAE
We use the optimized CatBoost as the new best candidate.

### If Optuna improves validation but not test
That is a warning that the optimization may be overfitting the validation period. We should not automatically use it.

### If Optuna does not improve either
Keep the previous CatBoost configuration and focus on better feature engineering.

**Do not run more optimization based on the test score.** The test set remains the final unbiased evaluation.